# Hand gesture recognition — CNN version (GPU)

Companion to `train.py` (HOG + SVM, CPU). Trains a small CNN on the **leapGestRecog** dataset
with a **leave-people-out split** (subjects 08 and 09 are never seen during training), so the
reported accuracy reflects new users, not memorised hands.

**Run it on Kaggle:** New Notebook → *Add data* → search `leapgestrecog` → Settings → Accelerator: GPU → Run all.
On Colab, upload the dataset and change `DATA` below.

In [ ]:
import os, glob, numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

candidates = glob.glob('/kaggle/input/**/leapGestRecog', recursive=True)
DATA = candidates[0] if candidates else 'leapGestRecog'   # <- change if needed
IMG_H, IMG_W, BATCH, SEED = 96, 256, 64, 42
GESTURES = ['palm','l','fist','fist_moved','thumb','index','ok','palm_moved','c','down']
TEST_SUBJECTS = {'08', '09'}
tf.keras.utils.set_random_seed(SEED)
print(DATA, tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# The dataset is <subject>/<NN_gesture>/<frame>.png  — there are NO train/validation/test folders,
# so we build the split ourselves, grouped by subject.
paths, labels, subjects = [], [], []
for p in sorted(glob.glob(os.path.join(DATA, '*', '*', '*.png'))):
    subj, gest = p.split(os.sep)[-3], p.split(os.sep)[-2]
    paths.append(p); labels.append(int(gest.split('_')[0]) - 1); subjects.append(subj)
paths, labels, subjects = np.array(paths), np.array(labels), np.array(subjects)
test_mask = np.isin(subjects, list(TEST_SUBJECTS))
rng = np.random.default_rng(SEED)
train_idx = np.where(~test_mask)[0]; rng.shuffle(train_idx)
n_val = int(0.15 * len(train_idx))
val_idx, train_idx, test_idx = train_idx[:n_val], train_idx[n_val:], np.where(test_mask)[0]
print(len(train_idx), 'train /', len(val_idx), 'val /', len(test_idx), 'test (unseen people)')

def load(path, label):
    img = tf.io.decode_png(tf.io.read_file(path), channels=1)
    img = tf.image.resize(img, (IMG_H, IMG_W)) / 255.0
    return img, label

augment = tf.keras.Sequential([tf.keras.layers.RandomTranslation(0.1, 0.1),
                               tf.keras.layers.RandomZoom(0.1),
                               tf.keras.layers.RandomContrast(0.2)])
def ds(idx, train=False):
    d = tf.data.Dataset.from_tensor_slices((paths[idx], labels[idx]))
    if train: d = d.shuffle(len(idx), seed=SEED)
    d = d.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if train: d = d.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return d.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = ds(train_idx, True), ds(val_idx), ds(test_idx)

In [ ]:
L = tf.keras.layers
model = tf.keras.Sequential([
    L.Input((IMG_H, IMG_W, 1)),
    L.Conv2D(32, 3, padding='same', activation='relu'), L.BatchNormalization(), L.MaxPooling2D(),
    L.Conv2D(64, 3, padding='same', activation='relu'), L.BatchNormalization(), L.MaxPooling2D(),
    L.Conv2D(128, 3, padding='same', activation='relu'), L.BatchNormalization(), L.MaxPooling2D(),
    L.Conv2D(128, 3, padding='same', activation='relu'), L.GlobalAveragePooling2D(),
    L.Dropout(0.4), L.Dense(10, activation='softmax')])
model.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
      tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]
history = model.fit(train_ds, validation_data=val_ds, epochs=25, callbacks=cb)

In [ ]:
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(1); y_true = labels[test_idx]
print(f'Accuracy on UNSEEN people: {(y_pred == y_true).mean():.4f}')
print(classification_report(y_true, y_pred, target_names=GESTURES))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='train'); ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy'); ax[0].legend()
cm = confusion_matrix(y_true, y_pred, normalize='true')
ax[1].imshow(cm, cmap='Blues'); ax[1].set_xticks(range(10), GESTURES, rotation=45); ax[1].set_yticks(range(10), GESTURES)
ax[1].set_title('Confusion matrix (unseen people)'); plt.tight_layout(); plt.savefig('cnn_results.png'); plt.show()
model.save('hand_gesture_cnn.keras')